# CA-SCAM论文实验材料提取

固定 InceptionDW + DPLS + VGUP，对SCAM与CA-SCAM进行同协议逐图比较，导出真实局部对比度、空间校准、残差和beta证据。不会训练。

权重目录：`MyDrive/ship_detection/paper_project/论文实验材料GPU/weights/`

- `InceptionDW_DPLS_SCAM_VGUP.pt`
- `InceptionDW_DPLS_CA-SCAM_VGUP.pt`

In [ ]:
# 作用：挂载Google Drive、安装与正式checkpoint一致的Ultralytics版本，并取得分析代码。
from google.colab import drive
drive.mount('/content/drive')

import base64
import json
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     'ultralytics==8.4.92', 'tqdm', 'pyyaml', 'pandas', 'matplotlib'],
    check=True,
)

import ultralytics
assert ultralytics.__version__ == '8.4.92', ultralytics.__version__
print('Ultralytics:', ultralytics.__version__)

def resolve_github_token():
    # 统一解析GITHUB_TOKEN，不依赖任何前置Cell。
    value = os.environ.get('GITHUB_TOKEN', '').strip()
    source = '环境变量 GITHUB_TOKEN'
    secret_error = None

    if not value:
        try:
            from google.colab import userdata
            value = (userdata.get('GITHUB_TOKEN') or '').strip()
            source = 'Colab Secrets: GITHUB_TOKEN'
        except Exception as error:
            secret_error = f'{type(error).__name__}: {error}'

    if not value:
        if secret_error:
            print('未能读取Colab Secret GITHUB_TOKEN：', secret_error)
        print('改用getpass安全输入；输入内容不会显示，也不会写入Notebook。')
        value = getpass('GitHub Token: ').strip()
        source = 'getpass'

    if not value:
        raise RuntimeError('没有取得GITHUB_TOKEN，无法读取私有仓库。')

    os.environ['GITHUB_TOKEN'] = value
    return value, source


def validate_github_identity(github_token):
    # 验证Token本身有效；不打印Token、长度或前缀。
    request = Request(
        'https://api.github.com/user',
        headers={
            'Authorization': f'Bearer {github_token}',
            'Accept': 'application/vnd.github+json',
            'X-GitHub-Api-Version': '2022-11-28',
            'User-Agent': 'ship-yolo-colab-paper-artifacts',
        },
    )
    try:
        with urlopen(request, timeout=30) as response:
            profile = json.load(response)
    except HTTPError as error:
        if error.code == 401:
            raise RuntimeError('GitHub拒绝该Token（HTTP 401）：Token无效或已过期。') from None
        if error.code == 403:
            raise RuntimeError('GitHub接受了请求但拒绝访问（HTTP 403）：请检查Token权限或账号状态。') from None
        raise RuntimeError(f'GitHub API验证失败：HTTP {error.code}。') from None
    except URLError as error:
        raise RuntimeError(f'无法连接GitHub API：{error.reason}') from None
    return profile.get('login', '未知账号')


GITHUB_TOKEN, credential_source = resolve_github_token()
github_login = validate_github_identity(GITHUB_TOKEN)
print('GitHub身份验证成功：', github_login)
print('凭据来源：', credential_source)

REPO_DIR = Path('/content/ship-yolo')
REPO_URL = 'https://github.com/HoverdZ/ship-yolo.git'
BRANCH = 'paper/extract-experiment-materials'
auth = base64.b64encode(f'x-access-token:{GITHUB_TOKEN}'.encode()).decode()
git_env = os.environ.copy()
git_env.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}',
})

probe = subprocess.run(
    ['git', 'ls-remote', '--exit-code', REPO_URL, f'refs/heads/{BRANCH}'],
    capture_output=True,
    text=True,
    env=git_env,
)
if probe.returncode != 0:
    detail = (probe.stderr or probe.stdout or '无Git错误详情').strip().splitlines()[-1]
    raise RuntimeError(
        'GitHub Token本身有效，但无法读取私有仓库目标分支。'
        '请检查Token是否具有该仓库Contents: Read权限，以及账号是否仍可访问仓库。'
        f' Git返回：{detail}'
    )
print('私有仓库与目标分支读取验证成功：', BRANCH)

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'paper/extract-experiment-materials'], check=True, env=git_env)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', 'paper/extract-experiment-materials'], check=True)
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'paper/extract-experiment-materials'],
        check=True,
        env=git_env,
    )
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} 已存在但不是Git仓库，请人工确认；程序不会自动删除。')
    subprocess.run(
        ['git', 'clone', '--branch', 'paper/extract-experiment-materials', '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
        env=git_env,
    )

sys.path.insert(0, str(REPO_DIR))
print('仓库提交:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# 作用：多线程复制Drive数据集到Colab本地；文件数与字节数均实时显示，重复运行时只补齐变化文件。
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm
import yaml

DRIVE_DATA = Path('/content/drive/MyDrive/ship_detection/data')
LOCAL_DATA = Path('/content/ship_detection/data')
if not DRIVE_DATA.is_dir():
    raise FileNotFoundError(DRIVE_DATA)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

source_files = sorted(path for path in DRIVE_DATA.rglob('*') if path.is_file())
total_bytes = sum(path.stat().st_size for path in source_files)
workers = min(32, max(8, (os.cpu_count() or 4) * 4))

def copy_one(source):
    relative = source.relative_to(DRIVE_DATA)
    destination = LOCAL_DATA / relative
    size = source.stat().st_size
    if destination.is_file() and destination.stat().st_size == size:
        return size, False
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.part')
    shutil.copyfile(source, temporary)
    temporary.replace(destination)
    return size, True

copied = 0
with tqdm(total=len(source_files), desc='数据集文件', unit='个', dynamic_ncols=True) as file_bar,      tqdm(total=total_bytes, desc='数据集字节', unit='B', unit_scale=True, unit_divisor=1024, dynamic_ncols=True) as byte_bar,      ThreadPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(copy_one, path) for path in source_files]
    for future in as_completed(futures):
        size, changed = future.result()
        copied += int(changed)
        file_bar.update(1)
        byte_bar.update(size)
        file_bar.set_postfix(本次复制=copied, 已存在=len(source_files) - copied)

def split_images(root, names):
    candidates = []
    for name in names:
        candidates.extend([root / 'images' / name, root / name / 'images'])
    return next((path for path in candidates if path.is_dir()), None)

train_images = split_images(LOCAL_DATA, ['train'])
val_images = split_images(LOCAL_DATA, ['val', 'valid', 'validation'])
test_images = split_images(LOCAL_DATA, ['test'])
if train_images is None or val_images is None:
    raise RuntimeError(f'无法识别训练/验证图片目录：{LOCAL_DATA}')

LOCAL_YAML = Path('/content/ship_detection/data.yaml')
payload = {
    'path': str(LOCAL_DATA),
    'train': str(train_images.relative_to(LOCAL_DATA)),
    'val': str(val_images.relative_to(LOCAL_DATA)),
    'nc': 1,
    'names': {0: 'ship'},
}
if test_images is not None:
    payload['test'] = str(test_images.relative_to(LOCAL_DATA))
LOCAL_YAML.parent.mkdir(parents=True, exist_ok=True)
LOCAL_YAML.write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding='utf-8')

print('本地数据集:', LOCAL_DATA)
print('本地data.yaml:', LOCAL_YAML)
print(payload)

In [ ]:
# 作用：固定推理协议和Drive输出路径；本Notebook不训练任何模型。
from pathlib import Path
import torch

assert torch.cuda.is_available(), '本任务依赖GPU；请将Colab运行时切换为GPU。'
GPU_ROOT = Path('/content/drive/MyDrive/ship_detection/paper_project/论文实验材料GPU')
WEIGHT_DIR = GPU_ROOT / 'weights'
OUTPUT_ROOT = GPU_ROOT / '输出'
GPU_ROOT.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMGSZ = 640
CONFIDENCE_FLOOR = 0.001
COUNT_CONFIDENCE = 0.25
NMS_IOU = 0.7
DEVICE = 0
BATCH = 8
print('GPU:', torch.cuda.get_device_name(0))
print('输出根目录:', OUTPUT_ROOT)

SCAM_WEIGHT = WEIGHT_DIR / 'InceptionDW_DPLS_SCAM_VGUP.pt'
CA_SCAM_WEIGHT = WEIGHT_DIR / 'InceptionDW_DPLS_CA-SCAM_VGUP.pt'
for path in (SCAM_WEIGHT, CA_SCAM_WEIGHT):
    if not path.is_file():
        raise FileNotFoundError(path)
CA_OUTPUT = OUTPUT_ROOT / '02_CA-SCAM'
CA_OUTPUT.mkdir(parents=True, exist_ok=True)
print('CA-SCAM输出:', CA_OUTPUT)

In [ ]:
# 作用：生成SCAM/CA-SCAM预测缓存、代表案例和不改变forward的内部机制可视化。
import pandas as pd
from tools.paper_artifacts.gpu_material_pipeline import (
    compare_prediction_caches,
    export_ca_scam_debug,
    generate_prediction_cache,
    package_gpu_results,
)

def ensure_cache(label, weights, output_dir, split='val'):
    expected = output_dir / f'{label.replace("/", "_").replace(" ", "_")}_{split}逐图预测缓存.json'
    if expected.is_file():
        import json
        from tools.paper_artifacts.gpu_material_pipeline import sha256_file
        metadata = json.loads(expected.read_text(encoding='utf-8'))
        valid = (
            metadata.get('model') == label
            and metadata.get('split') == split
            and int(metadata.get('imgsz', -1)) == IMGSZ
            and float(metadata.get('confidence_floor', -1)) == CONFIDENCE_FLOOR
            and float(metadata.get('nms_iou', -1)) == NMS_IOU
            and metadata.get('weights_sha256') == sha256_file(weights)
        )
        if valid:
            print('复用已通过权重与协议审计的预测缓存:', expected)
            return expected
        print('缓存与当前权重或推理协议不一致，将重新生成:', expected)
    return generate_prediction_cache(
        weights=weights,
        data_yaml=LOCAL_YAML,
        output_dir=output_dir,
        model_label=label,
        split=split,
        imgsz=IMGSZ,
        confidence_floor=CONFIDENCE_FLOOR,
        nms_iou=NMS_IOU,
        device=DEVICE,
        batch=BATCH,
    )

scam_cache = ensure_cache('SCAM', SCAM_WEIGHT, CA_OUTPUT, split='val')
ca_cache = ensure_cache('CA-SCAM', CA_SCAM_WEIGHT, CA_OUTPUT, split='val')
if test_images is not None:
    ensure_cache('SCAM', SCAM_WEIGHT, CA_OUTPUT, split='test')
    ensure_cache('CA-SCAM', CA_SCAM_WEIGHT, CA_OUTPUT, split='test')
    print('test逐图缓存已生成；代表案例筛选仍只使用val。')
candidate_csv = compare_prediction_caches(
    left_cache=scam_cache,
    right_cache=ca_cache,
    output_dir=CA_OUTPUT,
    prefix='CA-SCAM',
    left_label='SCAM',
    right_label='CA-SCAM',
    confidence_threshold=COUNT_CONFIDENCE,
)
candidates = pd.read_csv(candidate_csv)
example_image = candidates.iloc[0]['源路径']
export_ca_scam_debug(
    weights=CA_SCAM_WEIGHT,
    image=example_image,
    output_dir=CA_OUTPUT,
    imgsz=IMGSZ,
    device=DEVICE,
)
zip_path = package_gpu_results(OUTPUT_ROOT, GPU_ROOT / '论文实验材料_GPU结果.zip')
print('CA-SCAM材料完成:', CA_OUTPUT)
print('结果ZIP:', zip_path)